***5C) SEVERITY CORRELATION (per-interaction Spearman vs SLEDAI, FDR-corrected)***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "06A_GSE135779_SEVERITY_CORRELATION")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from statistical_utils import approximate_spearman_ci

metadata = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/patient_clinical_metadata.csv")

def correlate_with_severity(scores_path, out_path, min_patients=5):
    scores = pd.read_csv(scores_path)
    merged = scores.merge(metadata[["sample", "SLEDAI"]], on="sample", how="left")
    merged = merged.dropna(subset=["SLEDAI", "score"])

    records = []
    for interaction_id, group in merged.groupby("interaction_id"):
        if group["sample"].nunique() < min_patients:
            continue
        if group["score"].std() == 0:
            continue
        rho, pval = spearmanr(group["score"], group["SLEDAI"])
        ci_low, ci_high = approximate_spearman_ci(
            rho, group["sample"].nunique()
        )
        row = group.iloc[0]
        records.append({
            "interaction_id": interaction_id,
            "source": row["source"], "target": row["target"],
            "ligand_complex": row["ligand_complex"], "receptor_complex": row["receptor_complex"],
            "n_patients": group["sample"].nunique(),
            "spearman_rho": rho, "spearman_ci95_low": ci_low,
            "spearman_ci95_high": ci_high, "p_value": pval,
        })

    res = pd.DataFrame(records)
    if len(res) > 0:
        res["fdr_q_value"] = multipletests(res["p_value"], method="fdr_bh")[1]
        res = res.sort_values("fdr_q_value")
    res.to_csv(out_path, index=False)
    print(f"  Tested {len(res)} interactions")
    print(f"  Significant at FDR < 0.05: {(res['fdr_q_value'] < 0.05).sum() if len(res) else 0}")
    print(f"  Significant at FDR < 0.10: {(res['fdr_q_value'] < 0.10).sum() if len(res) else 0}")
    return res

In [ ]:
print("=== CHILD (cSLE): interaction score vs SLEDAI ===")
child_corr = correlate_with_severity(
    scores_path=f"{BASE_DIR}/Results/severity_analysis/cSLE_per_patient_scores.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/cSLE_severity_correlation.csv",
)
child_corr.head(20)

In [ ]:
print("=== ADULT (aSLE): interaction score vs SLEDAI ===")
print("Note: only 7 adult SLE patients have SLEDAI -- statistical power here is inherently low.")
adult_corr = correlate_with_severity(
    scores_path=f"{BASE_DIR}/Results/severity_analysis/aSLE_per_patient_scores.csv",
    out_path=f"{BASE_DIR}/Results/severity_analysis/aSLE_severity_correlation.csv",
    min_patients=5,
)
adult_corr.head(20)

**Severity correlation volcano plots.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, df, label in [(axes[0], child_corr, "cSLE"), (axes[1], adult_corr, "aSLE")]:
    sig = df["fdr_q_value"] < 0.05
    ax.scatter(df.loc[~sig, "spearman_rho"], -np.log10(df.loc[~sig, "p_value"]),
               s=6, alpha=0.3, color="gray", label="not significant")
    ax.scatter(df.loc[sig, "spearman_rho"], -np.log10(df.loc[sig, "p_value"]),
               s=8, alpha=0.7, color="#A6392A", label="FDR < 0.05")
    ax.set_xlabel("Spearman rho (score vs SLEDAI)")
    ax.set_ylabel("-log10(p-value)")
    ax.set_title(f"{label}: {sig.sum()}/{len(df)} interactions significant at FDR<0.05")
    ax.legend(fontsize=8)
plt.tight_layout()
out_path = f"{BASE_DIR}/Results/severity_analysis/severity_correlation_volcano.png"
plt.savefig(out_path, dpi=600)
plt.show()
print("Saved:", out_path)